In [20]:
# ===========================
# 01. IMPORTS
# ===========================
import re
from delta.tables import DeltaTable


# ===========================
# 02. PARAMETERS
# ===========================

# Default parameter values
DEFAULT_SOURCE_CATALOG = "lab_catalog_bronze"
DEFAULT_DEST_CATALOG   = "lab_catalog_silver"
DEFAULT_SCHEMA         = "digital_insurance"
DEFAULT_TABLE          = "tb_fraude_sinal"
DEFAULT_PK_FIELD_OLD   = "id"
DEFAULT_PK_FIELD       = "fraude_sinal_id"

# Read input parameters with default fallback values
source_catalog = oidlUtils.parameters.getParameter("source_catalog", DEFAULT_SOURCE_CATALOG)
dest_catalog   = oidlUtils.parameters.getParameter("dest_catalog", DEFAULT_DEST_CATALOG)
schema         = oidlUtils.parameters.getParameter("schema", DEFAULT_SCHEMA)
table          = oidlUtils.parameters.getParameter("table", DEFAULT_TABLE)
pk_field_old   = oidlUtils.parameters.getParameter("pk_field_old", DEFAULT_PK_FIELD_OLD)
pk_field       = oidlUtils.parameters.getParameter("pk_field", DEFAULT_PK_FIELD)

# Build derived paths and table names
source_full_table_name = f"{source_catalog}.{schema}.{table}"
dest_full_table_name   = f"{dest_catalog}.{schema}.{table}"
silver_checkpoint_path = f"/Volumes/{dest_catalog}/{schema}/checkpoints/{table}_checkpoint_silver/"
query_path             = f"/Workspace/ai-data-platform/src/silver-ingestion/queries/{table}.sql"

In [39]:
# ===========================
# 03. HELPERS
# ===========================
# Append CDF metadata fields to the business query.
CDF_FIELDS = ["_change_type"]

def import_query(path):
    """Load the SQL query from a file."""
    with open(path, "r") as open_file:
        return open_file.read()


def format_query_cdf(sql, source):

    select, from_clause = re.split(r"(?is)\bFROM\b", sql, maxsplit=1)
    extra_fields = ",\n    ".join(CDF_FIELDS)

    # Replace the original source with the deduplicated CDF dataset.
    from_clause = re.sub(r"^\s*\S+", f" (\n{source}\n) src", from_clause, count=1)

    return f"{select.strip()},\n    {extra_fields}\nFROM{from_clause}"


def upsert(df, delta_table_bronze, batch_id):
    count = df.count()

    if count == 0:
        print(f"[batch {batch_id}] No records to process.")
        return

    # Register the change feed micro-batch as a temporary view.
    df.createOrReplaceGlobalTempView(f"view_{table}")

    # Keep only the latest change for each source primary key.
    last_value = f"""
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY {pk_field_old}
                       ORDER BY _commit_timestamp DESC
                   ) AS rn
            FROM global_temp.view_{table}
            WHERE _change_type <> 'update_preimage'
        ) t
        WHERE rn = 1
    """

    # Build the business query using the deduplicated CDF dataset.
    query = format_query_cdf(import_query(query_path).strip(), last_value)
    df_upsert = spark.sql(query)

    # Exclude CDF control fields from update and insert operations.
    set_expr = {
        c: f"d.{c}"
        for c in df_upsert.columns
        if c not in CDF_FIELDS
    }

    # Apply CDF changes to the target Delta table.
    (
        delta_table_silver.alias("s")
        .merge(df_upsert.alias("d"), f"s.{pk_field} = d.{pk_field}")
        .whenMatchedDelete(condition="d._change_type = 'delete'")
        .whenMatchedUpdate(
            condition="d._change_type = 'update_postimage'",
            set=set_expr,
        )
        .whenNotMatchedInsert(
            condition="d._change_type IN ('insert', 'update_postimage')",
            values=set_expr,
        )
        .execute()
    )

    print(f"[batch {batch_id}] {dest_full_table_name} table updated - {count} records processed.")

In [40]:
# ===========================
# 04. FULL LOAD INGESTION
# ===========================

# Create the target Delta table only if it does not already exist.
if not spark.catalog.tableExists(dest_full_table_name):
    print("Target table does not exist. Starting full load...")

    # Load the business query from the SQL file.
    query = import_query(query_path).strip()
    if not query:
        raise ValueError(f"SQL file is empty: {query_path}")

    # Execute the initial load and create the target Delta table.
    (
        spark.sql(query)
          .write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(dest_full_table_name)
    )
    print("Full load completed successfully.")
else:
    print("Target table already exists. Skipping full load.")

Target table already exists. Skipping full load.


In [41]:
# ===========================
# 05. INCREMENTAL CDF PROCESSING
# ===========================

# Reference the target Delta table.
delta_table_silver = DeltaTable.forName(spark, dest_full_table_name)

# Get the physical location of the source Delta table.
table_path = (
    spark.sql(f"DESCRIBE DETAIL {source_full_table_name}")
         .select("location")
         .first()["location"]
)

# Create a streaming DataFrame from the Delta Change Data Feed.
df = (
    spark.readStream
        .format("delta")
        .option("readChangeFeed", "true")
        .load(table_path)
)

# Process each micro-batch and apply CDF changes to the target table.
stream = (
    df.writeStream
      .option("checkpointLocation", silver_checkpoint_path)
      .foreachBatch(lambda batch_df, batch_id: upsert(batch_df, delta_table_silver))
      .trigger(availableNow=True)  # Change to processingTime="1 minute" for continuous execution every minute.
      .start()
)

# Wait for the streaming job to complete.
stream.awaitTermination()